In [ ]:
from functools import cache
from typing import Callable
from sage.combinat.partition import Partitions
from sage.all import ZZ, factorial, binomial, product, matrix, srange, oeis
from sage.all import rising_factorial, falling_factorial, bell_number, var
from sage.all import PolynomialRing, flatten

In VS Code set in settings: { "notebook.output.textLineLimit": 500 }

In [ ]:
def OEIS_ID(M, verbose=True, num=4) -> list[str]:
    global oeis_search_on
    if not oeis_search_on:
        return [""]
    L = [M[i, j] for i in range(M.nrows()) for j in range(i+1)]
    s = ""
    for v in L:
        s += v.str() + " "
    R = oeis(s, max_results=num)
    if verbose: print(R)
    A = [u.id() for u in R]
    if A == []: 
        L = [M[i, j] for i in range(1, M.nrows()) for j in range(1, i+1)]
        s = ""
        for v in L:
            s += v.str() + " "
        R = oeis(s, max_results=num)
        if verbose: print(R)
        A = [u.id() for u in R]
    if A == []:
        if verbose: print("\nConsider to submit to the OEIS!")
    return A

The transformation $B: G \to T$ can be described by the following explicit closed formula:

$$T(n, k) = \frac{n!}{k!} \sum_{j_1 + \dots + j_k = n, j_i \ge 1} \prod_{i=1}^k \frac{G(j_i)}{j_i!}$$

Alternatively, using the *Faà di Bruno's formula*, it can be written as a sum over the set of partitions of $n$ into $k$ parts:

$$T(n, k) = \sum_{\pi \in \text{Part}(n, k)} \frac{n!}{\prod_{p \in \pi} (\text{size}(p)!)} \prod_{p \in \pi} G(\text{size}(p))$$

Some background can be found on
http://oeis.org/wiki/User:Peter_Luschny/BellTransform

In [ ]:
def bell_transform(n: int, a: list) -> list:
    # Precompute factorials 0..n once.
    fact = [factorial(i) for i in range(n + 1)]
    fn = fact[n]
    row = []
    for k in range(n + 1):
        result = 0
        for p in Partitions(n, length=k):
            denom = 1
            a_prod = 1
            # Single pass over the exponential dictionary:
            for part, cnt in p.to_exp_dict().items():
                denom *= fact[cnt] * fact[part] ** cnt
                a_prod *= a[part - 1] ** cnt
            result += (fn // denom) * a_prod
        row.append(result)
    return row

In [52]:
for n in range(8):
    print(bell_transform(n, [1, 1, 2, 6, 24, 120, 720, 5040]))

[1]
[0, 1]
[0, 1, 1]
[0, 2, 3, 1]
[0, 6, 11, 6, 1]
[0, 24, 50, 35, 10, 1]
[0, 120, 274, 225, 85, 15, 1]
[0, 720, 1764, 1624, 735, 175, 21, 1]


In [ ]:
def bell_matrix(generator: Callable, dim: int) -> matrix[ZZ]:
    G = [generator(k) for k in srange(dim)]
    row = lambda n: bell_transform(n, G)
    return matrix(ZZ, [row(n) + [0] * (dim - n - 1) for n in srange(dim)])

In [54]:
B = bell_matrix(factorial, 8)
print(B)
print(OEIS_ID(B))

[   1    0    0    0    0    0    0    0]
[   0    1    0    0    0    0    0    0]
[   0    1    1    0    0    0    0    0]
[   0    2    3    1    0    0    0    0]
[   0    6   11    6    1    0    0    0]
[   0   24   50   35   10    1    0    0]
[   0  120  274  225   85   15    1    0]
[   0  720 1764 1624  735  175   21    1]


0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']


The last result can be phrased:

The Bell transform of the factorial numbers A000142 is the triangle of the unsigned Stirling numbers of the first kind A132393.

In [55]:
# The inverse matrix.
B = bell_matrix(factorial, 8).inverse()
print(B)
print(OEIS_ID(B))

[   1    0    0    0    0    0    0    0]
[   0    1    0    0    0    0    0    0]
[   0   -1    1    0    0    0    0    0]
[   0    1   -3    1    0    0    0    0]
[   0   -1    7   -6    1    0    0    0]
[   0    1  -15   25  -10    1    0    0]
[   0   -1   31  -90   65  -15    1    0]
[   0    1  -63  301 -350  140  -21    1]
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
['A048993']


The last result can be phrased:

The inverse matrix of the Bell transform of the factorial numbers A000142 is the triangle of the signed Stirling numbers of the second kind, A048993 with signs.

In [ ]:
# The inverse transform.
def inverse_bell_matrix(generator: Callable, dim: int) -> matrix[ZZ]:
    M = bell_matrix(generator, dim).inverse()
    return matrix(ZZ, dim, lambda n, k: (-1)^(n - k) * M[n, k])

In [ ]:
B = inverse_bell_matrix(factorial, 8)
print(B)
print(OEIS_ID(B))

[  1   0   0   0   0   0   0   0]
[  0   1   0   0   0   0   0   0]
[  0   1   1   0   0   0   0   0]
[  0   1   3   1   0   0   0   0]
[  0   1   7   6   1   0   0   0]
[  0   1  15  25  10   1   0   0]
[  0   1  31  90  65  15   1   0]
[  0   1  63 301 350 140  21   1]
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
['A048993']


The last result can be phrased:

The inverse Bell transform of the factorial numbers A000142 is the triangle of the Stirling numbers of the second kind A048993.

In [58]:
B = bell_matrix(lambda n: n + 1, 8).inverse()
print(B)
print(OEIS_ID(B))

[      1       0       0       0       0       0       0       0]
[      0       1       0       0       0       0       0       0]
[      0      -2       1       0       0       0       0       0]
[      0       9      -6       1       0       0       0       0]
[      0     -64      48     -12       1       0       0       0]
[      0     625    -500     150     -20       1       0       0]
[      0   -7776    6480   -2160     360     -30       1       0]
[      0  117649 -100842   36015   -6860     735     -42       1]
0: A137452: Triangular array of the coefficients of the sequence of Abel polynomials A(n,x) := x*(x-n)^(n-1).
['A137452']


In [59]:
B = bell_matrix(lambda n: (-n - 1)^n, 8)
print(B)
print(OEIS_ID(B))

[      1       0       0       0       0       0       0       0]
[      0       1       0       0       0       0       0       0]
[      0      -2       1       0       0       0       0       0]
[      0       9      -6       1       0       0       0       0]
[      0     -64      48     -12       1       0       0       0]
[      0     625    -500     150     -20       1       0       0]
[      0   -7776    6480   -2160     360     -30       1       0]
[      0  117649 -100842   36015   -6860     735     -42       1]
0: A137452: Triangular array of the coefficients of the sequence of Abel polynomials A(n,x) := x*(x-n)^(n-1).
['A137452']


The last two results may be phrased:

The coefficients of the Abel polynomials A137452 are given by the Bell transform of n -> (-n-1)^n, or, by the matrix inverse of the Bell transform of n -> n + 1.

In [60]:
B = inverse_bell_matrix(lambda n: n + 1, 8)
print(B)
print(OEIS_ID(B))

[     1      0      0      0      0      0      0      0]
[     0      1      0      0      0      0      0      0]
[     0      2      1      0      0      0      0      0]
[     0      9      6      1      0      0      0      0]
[     0     64     48     12      1      0      0      0]
[     0    625    500    150     20      1      0      0]
[     0   7776   6480   2160    360     30      1      0]
[     0 117649 100842  36015   6860    735     42      1]
0: A137452: Triangular array of the coefficients of the sequence of Abel polynomials A(n,x) := x*(x-n)^(n-1).
['A137452']


In [61]:
B = bell_matrix(lambda n: n + 1, 8)
print(B)
print(OEIS_ID(B))

[   1    0    0    0    0    0    0    0]
[   0    1    0    0    0    0    0    0]
[   0    2    1    0    0    0    0    0]
[   0    3    6    1    0    0    0    0]
[   0    4   24   12    1    0    0    0]
[   0    5   80   90   20    1    0    0]
[   0    6  240  540  240   30    1    0]
[   0    7  672 2835 2240  525   42    1]
0: A059297: Triangle of idempotent numbers binomial(n,k)*k^(n-k), version 1.
['A059297']


This is A059297, the triangle of the idempotent numbers I(n, k) = binomial(n,k)*k^(n-k).
It is the matrix inverse of  A137452. 

In [62]:
# [A023531, A048993, A075497, A075498, A075499, A075500, ...]
for n in range(7):
    B = bell_matrix(lambda k: n^k, 7)
    print("\nBell matrix generated by power function", n)
    print(OEIS_ID(B))
    print(B)


Bell matrix generated by power function 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Bell matrix generated by power function 1
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
1: A151511: The triangle in A151359 read by rows downwards.
['A048993', 'A151511']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  1  3  1  0  0  0]
[ 0  1  7  6  1  0  0]
[ 0  1 15 25 10  1  0]
[ 0  1 31 90 65 15  1]

Bell matrix generated by power function 2

0: A075497: Stirling2 triangle with scaled diagonals (po

In [63]:
for n in range(7):
    B = inverse_bell_matrix(lambda k: n^k, 7)
    print("\nInverse Bell matrix generated by power function", n)
    print(OEIS_ID(B))
    print(B)


Inverse Bell matrix generated by power function 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Inverse Bell matrix generated by power function 1
0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   2   3   1   0   0   0]
[  0   6  11   6   1   0   0]
[  0  24  50  35  10   1   0]
[ 

In [64]:
multifactorial = lambda a, b: lambda n: product(a * k + b for k in range(n))

In [65]:
[multifactorial(2, 2)(n) for n in range(8)]

[1, 2, 8, 48, 384, 3840, 46080, 645120]

In [66]:
# A132393,
for a in range(1, 5):
    for b in range(1, a + 1):
        B = bell_matrix(multifactorial(a, b), 7)
        print("\nBell matrix generated by multifactorial", (a, b))
        print(OEIS_ID(B))
        print(B)



Bell matrix generated by multifactorial (1, 1)
0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   2   3   1   0   0   0]
[  0   6  11   6   1   0   0]
[  0  24  50  35  10   1   0]
[  0 120 274 225  85  15   1]

Bell matrix generated by multifactorial (2, 1)
0: A132062: Sheffer triangle (1,1-sqrt(1-2*x)). Extended Bessel triangle A001497.
1: A122850: Exponential Riordan array (1, sqrt(1+2x)-1).
['A132062', 'A122850']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   3   3   1   0   0   0]
[  0  15  15   6   1   0   0]
[  0 105 105  45  10   1   0]
[  0 945 945 420 105  15   1]

Bell matrix generated by multifactorial (2, 2)

0: A039683: Signed double Pochhammer triangl

In [67]:
for a in range(1, 5):
    for b in range(1, a + 1):
        B = inverse_bell_matrix(multifactorial(a, b), 7)
        print("\nInverse Bell matrix generated by multifactorial", (a, b))
        print(OEIS_ID(B))
        print(B)


Inverse Bell matrix generated by multifactorial (1, 1)
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
1: A151511: The triangle in A151359 read by rows downwards.
['A048993', 'A151511']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  1  3  1  0  0  0]
[ 0  1  7  6  1  0  0]
[ 0  1 15 25 10  1  0]
[ 0  1 31 90 65 15  1]

Inverse Bell matrix generated by multifactorial (2, 1)
0: A122848: Exponential Riordan array (1, x(1+x/2)).
['A122848']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  0  3  1  0  0  0]
[ 0  0  3  6  1  0  0]
[ 0  0  0 15 10  1  0]
[ 0  0  0 15 45 15  1]

Inverse Bell matrix generated by multifactorial (2, 2)

0: A075497: Stirling2 triangle with scaled diagonals (powers of 2).
['A075497']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   2   1   0   0   0   0]
[  0   4   6   1   0   0   0]
[  0   8  28  12   1   0   0]
[  0  16 120 100  20   1   0]
[  0  32 496 720 2

In [68]:
shifted_factorial = lambda sh: lambda n: factorial(n + sh)

In [69]:
# [A132393, A111596 (Lah), A136656, ...]
for sh in range(4):
    B = bell_matrix(shifted_factorial(sh), 7)
    print("\nBell matrix generated by shifted factorial", sh)
    print(OEIS_ID(B))
    print(B)


Bell matrix generated by shifted factorial 0
0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   2   3   1   0   0   0]
[  0   6  11   6   1   0   0]
[  0  24  50  35  10   1   0]
[  0 120 274 225  85  15   1]

Bell matrix generated by shifted factorial 1
0: A271703: Triangle read by rows: the unsigned Lah numbers T(n, k) = binomial(n-1, k-1)*n!/k! if n > 0 and k > 0, T(n, 0) = 0^n and otherwise 0, for n >= 0 and 0 <= k <= n.
1: A111596: The matrix inverse of the unsigned Lah numbers A271703.
['A271703', 'A111596']
[   1    0    0    0    0    0    0]
[   0    1    0    0    0    0    0]
[   0    2    1    0    0    0    0]
[   0    6    6    1    0    0    0]
[   0   24   36   12    1    0    0]
[   0  120  240  12

In [70]:
# A048993, A111596 (Lah),
for sh in range(2):
    B = inverse_bell_matrix(shifted_factorial(sh), 7)
    print("\nInverse Bell matrix generated by shifted factorial", sh)
    print(OEIS_ID(B))
    print(B)


Inverse Bell matrix generated by shifted factorial 0
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
1: A151511: The triangle in A151359 read by rows downwards.
['A048993', 'A151511']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  1  3  1  0  0  0]
[ 0  1  7  6  1  0  0]
[ 0  1 15 25 10  1  0]
[ 0  1 31 90 65 15  1]

Inverse Bell matrix generated by shifted factorial 1
0: A271703: Triangle read by rows: the unsigned Lah numbers T(n, k) = binomial(n-1, k-1)*n!/k! if n > 0 and k > 0, T(n, 0) = 0^n and otherwise 0, for n >= 0 and 0 <= k <= n.
1: A111596: The matrix inverse of the unsigned Lah numbers A271703.
['A271703', 'A111596']
[   1    0    0    0    0    0    0]
[   0    1    0    0    0    0    0]
[   0    2    1    0    0    0    0]
[   0    6    6    1    0    0    0]
[   0   24   36   12    1    0    0]
[   0  120  240  120   20    1    0]
[   0  720 1800 1200  300   30    1]


In [71]:
risingfactorial = lambda n: lambda k: rising_factorial(n, k)

In [72]:
# A023531, A132393, A111596, A046089, A049352, A049353, A049374, A134141, ...
for n in range(8):
    B = bell_matrix(risingfactorial(n), 7)
    print("\nBell matrix generated by rising factorial(n, k)", n)
    print(OEIS_ID(B))
    print(B)


Bell matrix generated by rising factorial(n, k) 0


0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Bell matrix generated by rising factorial(n, k) 1
0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   2   3   1   0   0   0]
[  0   6  11   6   1   0   0]
[  0  24  50  35  10   1   0]
[  0 120 274 225  85  15   1]

Bell matrix generated 

In [73]:
for n in range(8):
    B = inverse_bell_matrix(risingfactorial(n), 7)
    print("\nInverse Bell matrix generated by rising factorial(n, k)", n)
    print(OEIS_ID(B))
    print(B)


Inverse Bell matrix generated by rising factorial(n, k) 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Inverse Bell matrix generated by rising factorial(n, k) 1
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
1: A151511: The triangle in A151359 read by rows downwards.
['A048993', 'A151511']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  1  3  1  0  0  0]
[ 0  1  7  6  1  0  0]
[ 0  1 15 25 10  1  0]
[ 0  1 31 90 65 15  1]

Inverse Bell matrix generated by rising factorial(n, k) 2
0: A27170

In [74]:
monotonicfactorial = lambda r: lambda n: rising_factorial(r, n) // factorial(n)

In [75]:
for n in range(5):
    B = bell_matrix(monotonicfactorial(n), 7)
    print("\nBell matrix generated by monotonic factorial(r, n)", n)
    print(OEIS_ID(B))
    print(B)



Bell matrix generated by monotonic factorial(r, n) 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Bell matrix generated by monotonic factorial(r, n) 1
0: A048993: Triangle of Stirling numbers of 2nd kind, S(n,k), n >= 0, 0 <= k <= n.
1: A151511: The triangle in A151359 read by rows downwards.
['A048993', 'A151511']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  1  3  1  0  0  0]
[ 0  1  7  6  1  0  0]
[ 0  1 15 25 10  1  0]
[ 0  1 31 90 65 15  1]

Bell matrix generated by monotonic factorial(r, n) 2
0: A059297: Triangle of 

In [ ]:
# A061356
for n in range(5):
    B = inverse_bell_matrix(monotonicfactorial(n), 7)
    print("\nInverse Bell matrix generated by monotonic factorial(r, n)", n)
    print(OEIS_ID(B))
    print(B)


Inverse Bell matrix generated by monotonic factorial(r, n) 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Inverse Bell matrix generated by monotonic factorial(r, n) 1
0: A132393: Triangle of unsigned Stirling numbers of the first kind (see A048994), read by rows, T(n,k) for 0 <= k <= n.
1: A048994: Triangle of Stirling numbers of first kind, s(n,k), n >= 0, 0 <= k <= n.
['A132393', 'A048994']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   2   3   1   0   0   0]
[  0   6  11   6   1   0   0]
[  0  24  

In [ ]:
fallingfactorial = lambda n: lambda k: falling_factorial(n, k)

In [ ]:
# A023531, (A049403, A122848), A049404, A049410, A049424, A049411, ....
for n in range(6):
    B = bell_matrix(fallingfactorial(n), 7)
    print("\nBell matrix generated by falling factorial(n, k)", n)
    print(OEIS_ID(B))
    print(B)


Bell matrix generated by falling factorial(n, k) 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Bell matrix generated by falling factorial(n, k) 1
0: A122848: Exponential Riordan array (1, x(1+x/2)).
['A122848']
[ 1  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0]
[ 0  1  1  0  0  0  0]
[ 0  0  3  1  0  0  0]
[ 0  0  3  6  1  0  0]
[ 0  0  0 15 10  1  0]
[ 0  0  0 15 45 15  1]

Bell matrix generated by falling factorial(n, k) 2

0: A049404: Triangle read by rows, the Bell transform of n!*binomial(2,n) (without column 0).
['A049404']
[  1   0   0   0   0   0

In [ ]:
for n in range(8):
    B = inverse_bell_matrix(fallingfactorial(n), 7)
    print("\nInverse Bell matrix generated by falling factorial(n, k)", n)
    print(OEIS_ID(B))
    print(B)


Inverse Bell matrix generated by falling factorial(n, k) 0
0: A010054: a(n) = 1 if n is a triangular number, otherwise 0.
1: A023531: a(n) = 1 if n is of the form m(m+3)/2, otherwise 0.
2: A073424: Triangle read by rows: T(m,n) = parity of 0^n + 0^m, n = 0,1,2,3 ..., m = 0,1,2,3, ... n.
3: A073423: Sums of two powers of zero: triangle read by rows: T(m,n) = 0^n + 0^m, n >= 0, m = 0..n.
['A010054', 'A023531', 'A073424', 'A073423']
[1 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 1 0 0 0 0]
[0 0 0 1 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 0 0 0 0 1]

Inverse Bell matrix generated by falling factorial(n, k) 1
0: A132062: Sheffer triangle (1,1-sqrt(1-2*x)). Extended Bessel triangle A001497.
1: A122850: Exponential Riordan array (1, sqrt(1+2x)-1).
['A132062', 'A122850']
[  1   0   0   0   0   0   0]
[  0   1   0   0   0   0   0]
[  0   1   1   0   0   0   0]
[  0   3   3   1   0   0   0]
[  0  15  15   6   1   0   0]
[  0 105 105  45  10   1   0]
[  0 945 945 420 105  15   1]

Inverse Bell matrix 

In [80]:
def bell_second_order(generator, n):
    G = [generator(k) for k in range(n)]
    row = lambda n: bell_transform(n, G)
    S = [sum(row(k)) for k in range(n)]
    return bell_transform(n, S)

In [81]:
# A264430
[bell_second_order(bell_number, n) for n in range(8)]

[[1],
 [0, 1],
 [0, 1, 1],
 [0, 2, 3, 1],
 [0, 6, 11, 6, 1],
 [0, 23, 50, 35, 10, 1],
 [0, 106, 268, 225, 85, 15, 1],
 [0, 568, 1645, 1603, 735, 175, 21, 1]]

In [82]:
# A187761
[sum(s) for s in [bell_second_order(lambda k: 1, n) for n in range(10)]]

[1, 1, 2, 6, 23, 106, 568, 3459, 23544, 176850]

In [83]:
# A265023
[sum(s) for s in [bell_second_order(lambda k: (-1) ^ k, n) for n in range(10)]]

[1, 1, 2, 4, 9, 22, 54, 139, 372, 948]

In [84]:
# Generalizes A000248
[bell_second_order(lambda k: k + 1, n) for n in range(8)]

[[1],
 [0, 1],
 [0, 1, 1],
 [0, 3, 3, 1],
 [0, 10, 15, 6, 1],
 [0, 41, 80, 45, 10, 1],
 [0, 196, 486, 345, 105, 15, 1],
 [0, 1057, 3283, 2856, 1085, 210, 21, 1]]

See A256894.

In [85]:
@cache
def incomplete_bell_polynomial(n, k):
    Z = var(["z_" + str(i) for i in range(n - k + 1)])
    R = PolynomialRing(ZZ, Z, n - k + 1, order='lex')
    if k == 0:
        return R(k^n)
    return R(
        sum(
            binomial(n - 1, j - 1) * incomplete_bell_polynomial(n - j, k - 1) * Z[j - 1]
            for j in range(n - k + 2)
        ).expand()
    )

In [ ]:
def poly_row(n):
    return [incomplete_bell_polynomial(n, k) for k in range(n + 1)]

def coeff_row(n):
    return flatten([[0] if (c := p.coefficients()) == [] else c for p in poly_row(n)])

for n in range(6): print(poly_row(n))
for n in range(6): print(coeff_row(n))

[1]
[0, z_0]
[0, z_1, z_0^2]
[0, z_2, 3*z_0*z_1, z_0^3]
[0, z_3, 4*z_0*z_2 + 3*z_1^2, 6*z_0^2*z_1, z_0^4]
[0, z_4, 5*z_0*z_3 + 10*z_1*z_2, 10*z_0^2*z_2 + 15*z_0*z_1^2, 10*z_0^3*z_1, z_0^5]
[1]
[0, 1]
[0, 1, 1]
[0, 1, 3, 1]
[0, 1, 4, 3, 6, 1]
[0, 1, 5, 10, 10, 15, 10, 1]


See A356656.

In [87]:
def bell_polynomial(n):
    return sum(incomplete_bell_polynomial(n, k) for k in range(n + 1))


for n in range(6): print([n], bell_polynomial(n))

[0] 1
[1] z_0
[2] z_0^2 + z_1
[3] z_0^3 + 3*z_0*z_1 + z_2
[4] z_0^4 + 6*z_0^2*z_1 + 4*z_0*z_2 + 3*z_1^2 + z_3
[5] z_0^5 + 10*z_0^3*z_1 + 10*z_0^2*z_2 + 15*z_0*z_1^2 + 5*z_0*z_3 + 10*z_1*z_2 + z_4


In [100]:
for n in range(6): print([n], bell_polynomial(n).coefficients())

[0] [1]
[1] [1]
[2] [1, 1]
[3] [1, 3, 1]
[4] [1, 6, 4, 3, 1]
[5] [1, 10, 10, 15, 5, 10, 1]


In [89]:
def univariate_bell_polynomial(n):
    polynom = x^n
    fn = factorial(n)
    for k in range(n):
        result = 0
        for p in Partitions(n, length=k):
            factorial_product = 1
            power_factorial_product = 1
            for part, count in p.to_exp_dict().items():
                factorial_product *= factorial(count)
                power_factorial_product *= factorial(part) ** count
            coefficient = fn // (factorial_product * power_factorial_product)
            result += coefficient
        polynom += result * x^k
    return polynom


for n in range(7):
    print(univariate_bell_polynomial(n))

for n in range(7):
    print(univariate_bell_polynomial(n).list())

1
x
x^2 + x
x^3 + 3*x^2 + x
x^4 + 6*x^3 + 7*x^2 + x
x^5 + 10*x^4 + 25*x^3 + 15*x^2 + x
x^6 + 15*x^5 + 65*x^4 + 90*x^3 + 31*x^2 + x
[1]
[0, 1]
[0, 1, 1]
[0, 1, 3, 1]
[0, 1, 7, 6, 1]
[0, 1, 15, 25, 10, 1]
[0, 1, 31, 90, 65, 15, 1]


In [90]:
# Given a sequence f returns the inverse Bell sequence of f.
def Inverse_Bell_Sequence(f, dim):
    M = inverse_bell_matrix(f, dim)
    return [M[n][1] for n in (1..dim-1)]

In [ ]:
print(Inverse_Bell_Sequence(lambda n: 1, 9))

print(Inverse_Bell_Sequence(lambda n: factorial(n), 9))

print(Inverse_Bell_Sequence(lambda n: [1, 2, 2][n] if n < 3 else 0, 9))

print(Inverse_Bell_Sequence(lambda n: [1, 2, 1][n] if n < 3 else 0, 9))

[1, 1, 2, 6, 24, 120, 720, 5040]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 2, 10, 80, 880, 12320, 209440, 4188800]
[1, 2, 11, 100, 1270, 20720, 413000, 9726640]


In [92]:
def cb(n):
    return binomial(2 * n, n) 

print(Inverse_Bell_Sequence(cb, 20))

[1, 2, 6, 20, 50, -168, -4732, -54024, -356670, 1558040, 106069172, 2197188864, 26605890220, -22266781600, -12120090377400, -402165029201744, -7732409047854318, -38209542402620232, 4126723132306766900]


See A066397.

In [ ]:
def Bell_Matrix(generator, dim) -> list[list[int]]:
    A = [[0] * (n + 1) for n in range(dim)]
    for n in range(dim):
        A[n][0] = 1 if n == 0 else 0
        if n > 0:
            A[n][1] = generator(n - 1)
        for k in range(2, n + 1):
            A[n][k] = sum(
                binomial(n - 1, j - 1) * A[n - j][k - 1] * A[j][1]
                for j in range(1, n - k + 2)
            )
    return A

In [94]:
Bell_Matrix(factorial, 8)

[[1],
 [0, 1],
 [0, 1, 1],
 [0, 2, 3, 1],
 [0, 6, 11, 6, 1],
 [0, 24, 50, 35, 10, 1],
 [0, 120, 274, 225, 85, 15, 1],
 [0, 720, 1764, 1624, 735, 175, 21, 1]]

In [ ]:
def Inverse_Bell_Matrix(f, dim) -> list[list[int]]:
    A = Bell_Matrix(f, dim)
    M = [[0] * (n + 1) for n in range(dim)]
    for n in range(dim):
        M[n][n] = 1 / A[n][n]
        for k in range(n - 1, 0, -1):
            M[n][k] = -sum(A[i][k] * M[n][i] for i in range(n, k, -1)) / A[k][k]
    return M

In [ ]:
Inverse_Bell_Matrix(factorial, 7)

[[1],
 [0, 1],
 [0, -1, 1],
 [0, 1, -3, 1],
 [0, -1, 7, -6, 1],
 [0, 1, -15, 25, -10, 1],
 [0, -1, 31, -90, 65, -15, 1]]

https://trac.sagemath.org/ticket/18338